## **Autograd**

Version 1: Scalar Autgrad

In [83]:
import numpy as np
class Spyder:
    def __init__(self, data, parents = None):
        self.data = data
        self.grad = 0
        self.parents = parents if parents else []
        self._backward = lambda: None

    def __mul__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)
        out = Spyder(self.data * other.data, parents = [self, other])

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)
        out = Spyder(self.data + other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)
        out = Spyder(self.data - other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad
            other.grad -= out.grad

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float values supported"
        out = Spyder(self.data ** other, parents = [self,])

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __repr__(self):
        return f"Spyder(data: {self.data}, grad: {self.grad})"

    def retrace(self):
        self.clean_web()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited: 
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1

        for v in reversed(topo):
            v._backward()

    def clean_web(self):
        visited = set()
        def visit_children(v):
            if v not in visited: 
                visited.add(v)
                v.grad = 0

                for parent in v.parents:
                    visit_children(parent)
        visit_children(self)

In [84]:
x = Spyder(2)
a = x ** 3

In [85]:
a.retrace()
x.grad

12

Version 2: Primitive Tensor Autograd

In [86]:
class Spyder:
    def __init__(self, data: np.ndarray, parents = None):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.parents = parents if parents else []
        self._backward = lambda: None

    def __repr__(self):
        return f"Spyder(data = {self.data}, grad = {self.grad}, parents = {self.parents})"

    def __mul__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert self.data.shape == other.data.shape, "not supporting broadcasting for now"
        out = Spyder(self.data * other.data, parents=[self, other])

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __matmul__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert self.data.shape[1] == other.data.shape[0], "matrixes must be in shape (m, n), (n, p) for matmul"
        out = Spyder(self.data @ other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    def sum(self):
        out = Spyder(np.sum(self.data), parents = [self,])

        def _backward():
            self.grad += np.ones_like(self.data) * out.grad

        out._backward = _backward
        return out

    def mean(self):
        out = Spyder(np.mean(self.data), parents = [self])

        def _backward():
            self.grad += (
                np.ones_like(self.data) *
                out.grad /
                self.data.size
            )

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert (isinstance(other, (int, float))), "only supporting int/float powers for now"

        out = Spyder(self.data ** other, parents = [self]) 

        def _backward():
            self.grad += (other * (self.data) ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) + (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) + (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data + other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad += np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad += np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Spyder) else Spyder(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) + (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) + (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data - other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad -= np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad -= np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __neg__(self):
        out = Spyder(-self.data, parents = [self])
        def _backward():
            self.grad -= out.grad
        out._backward = _backward
        return out

    def __rsub__(self, other):
        assert isinstance(other, (int, float))

        other = Spyder(
            np.full(self.data.shape, other)
        )

        return other - self

    def __getitem__(self, idx):
        out = Spyder(self.data[idx], parents = [self])

        def _backward():
            self.grad[idx] += out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Spyder(np.maximum(0, self.data), parents = [self])

        def _backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = _backward
        return out

    def sigmoid(self):
        out = Spyder(1 / (1 + np.exp(-self.data)), parents = [self])

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad

        out._backward = _backward
        return out

    def log(self):
        out = Spyder(np.log(self.data), parents = [self])

        def _backward():
            self.grad += (1 / self.data) * out.grad

        out._backward = _backward
        return out
    
    def retrace(self):
        self.clean_webs()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

    def clean_webs(self):
        visited = set()
        def visit_parents(v):
            if v not in visited:
                visited.add(v)
                v.grad = np.zeros_like(v.data)
                for parent in v.parents:
                    visit_parents(parent)
        visit_parents(self)

Multi-layer Regression Neural Network

In [87]:
class Web:
    def __init__(self, layer_size, learning_rate = 0.001, intialization_strength = 0.01, random_state = 11, epochs = 1000):

        """
        x, y -> np.ndarray or spyder
        layer_size -> (input features, layer 1 neurons, layer 2 neurons, ..., outputs)
        """
        self.rng = np.random.default_rng(random_state)
        self.layer_size = layer_size
        self.learning_rate = learning_rate
        self.intialization_strength = intialization_strength
        self.epochs = epochs

    def spin(self, x, y):

        x = x if isinstance(x, Spyder) else Spyder(x)
        y = y if isinstance(y, Spyder) else Spyder(y)

        self.weights = []
        self.biases = []

        for n in range(len(self.layer_size) - 1):
        
            in_features = self.layer_size[n]
            out_features = self.layer_size[n + 1]
        
            self.weights.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (in_features, out_features))))
            self.biases.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (out_features,))))

        for epoch in range(self.epochs):
            inp = x
            for w, b in zip(self.weights[:-1], self.biases[:-1]):
                inp = (inp @ w + b).relu()

            pred = inp @ self.weights[-1] + self.biases[-1]
            error = ((pred - y) ** 2).mean()
            
            error.retrace()

            for w, b in zip(self.weights, self.biases):
                w.data -= self.learning_rate * w.grad
                b.data -= self.learning_rate * b.grad

            if epoch % 50 == 0:
                print(f"epoch {epoch}, loss = {error.data}")

    def predict(self, x):
        inp = x if isinstance(x, Spyder) else Spyder(x)
        for w, b in zip(self.weights[:-1], self.biases[:-1]):
            inp = (inp @ w + b).relu()
        pred = inp @ self.weights[-1] + self.biases[-1]
        return pred

In [88]:
np.array([2, 3, 4]).size

3

In [89]:
x = Spyder(np.array([[3.0]]))
w = Spyder(np.array([[2.0]]))

y = x @ w
loss = y.sum()

loss.retrace()

print(w.grad)

[[3.]]


In [90]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [91]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [92]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [93]:
x_train = transformations.fit_transform(x_train)

In [94]:
x_test = transformations.transform(x_test)

In [95]:
web = Web([x_train.shape[1],32, 16, 1], learning_rate = 0.05, intialization_strength=0.01, epochs = 1500)

In [96]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.to_numpy().reshape(-1, 1)
)

In [97]:
web.spin(x_train[:5000], y_train_scaled[:5000])

epoch 0, loss = 0.9903153545974539
epoch 50, loss = 0.9901224047944058
epoch 100, loss = 0.9901176155578978
epoch 150, loss = 0.9901126497029287
epoch 200, loss = 0.9901067905092604
epoch 250, loss = 0.9900992633700533
epoch 300, loss = 0.9900890228042717
epoch 350, loss = 0.990074469011847
epoch 400, loss = 0.9900529533257215
epoch 450, loss = 0.9900198607935865
epoch 500, loss = 0.9899664364748465
epoch 550, loss = 0.9898750973831524
epoch 600, loss = 0.9897056882408503
epoch 650, loss = 0.9893307795536295
epoch 700, loss = 0.9884982560916734
epoch 750, loss = 0.9862716444963658
epoch 800, loss = 0.9789632340933538
epoch 850, loss = 0.9475913874621398
epoch 900, loss = 0.7571758235760839
epoch 950, loss = 0.18679287902667988
epoch 1000, loss = 0.24534535852514672
epoch 1050, loss = 0.17915977706742797
epoch 1100, loss = 0.14236126251050432
epoch 1150, loss = 0.12059725693660757
epoch 1200, loss = 0.10636877145585154
epoch 1250, loss = 0.09514690260880251
epoch 1300, loss = 0.08614254

In [98]:
pred = web.predict(x_test)

In [99]:
train_pred = web.predict(x_train)
train_mse = np.mean((train_pred.data - y_train_scaled.reshape(-1,1))**2)

In [100]:
train_mse

np.float64(0.07217249108844467)

In [101]:
mse = np.mean((pred.data - y_test_scaled)**2)

In [102]:
mse

np.float64(0.07121071433434586)

Tensor Autograd but sum, mean and max are not axis aware.

In [103]:
class Spyder:
    def __init__(self, data: np.ndarray, parents = None):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.parents = parents if parents else []
        self._backward = lambda: None

    def _coerce(self, other):
        """function to coerce other into a spyder"""
        match other:
            case Spyder(): return other
            case isinstance(other, (numbers.Real, np.generic)): return Spyder(other) 
            case _: raise TypeError(
                f"Received object of type {type(other)}, was unable to coerce to Spyder object"
                f"\nOnly supporting scalars, np.arrays and other spyders!"
                )

    def __repr__(self):
        return f"Spyder(data = {self.data}, grad = {self.grad}, parents = {self.parents})"

    def __matmul__(self, other):
        other = self._coerce(other)

        try:
            assert self.data.shape[1] == other.data.shape[0], "matrixes must be in shape (m, n), (n, p) for matmul"
        except IndexError:
            raise ValueError("Sorry bro i was too lazy to implement any other broadcasting type :(\n"
                            f"So i got a matrices of shapes {self.data.shape} and {other.data.shape}"
                            "it is not in required (m, n), (n, p) form so i cannot calculate sorry :((\n"
                            "just use tinygrad or micrograd bro i so sorry pls forgive me")
        out = Spyder(self.data @ other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    def sum(self):
        out = Spyder(np.sum(self.data), parents = [self,])

        def _backward():
            self.grad += np.ones_like(self.data) * out.grad

        out._backward = _backward
        return out

    def mean(self):
        out = Spyder(np.mean(self.data), parents = [self])

        def _backward():
            self.grad += (
                np.ones_like(self.data) *
                out.grad /
                self.data.size
            )

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert (isinstance(other, (numbers.Real, np.generic))), "only supporting int/float powers for now"

        out = Spyder(self.data ** other, parents = [self]) 

        def _backward():
            self.grad += (other * (self.data) ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        if isinstance(other, (numbers.Real, np.generic)): # now we know that this isnt a spyder with a scalar value, its just a constant
            out = Spyder(self.data + other, parents=[self])

            def _backward():
                self.grad += out.grad

            out._backward = _backward

            return out
        
        other = self._coerce(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) + (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) + (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data + other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad += np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad += np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __sub__(self, other):
        if isinstance(other, (numbers.Real, np.generic)): # now we know that this isnt a spyder with a scalar value, its just a constant
            out = Spyder(self.data - other, parents=[self]) 
    
            def _backward():
                self.grad += out.grad 
                out._backward = _backward
        
            return out
                
        other = self._coerce(other)

        assert (
            self.data.shape == other.data.shape or # (m, n) - (m, n)
                (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) - (n,)
                ), "not supporting broadcasting for now"
        
        out = Spyder(self.data - other.data, parents=[self, other])

        def _backward():

            if self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad -= np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad -= np.sum(out.grad, axis = 0)

        out._backward = _backward
        return out

    def __mul__(self, other):
        if isinstance(other, (numbers.Real, np.generic)): # now we know that this isnt a spyder with a scalar value, its just a constant
            out = Spyder(self.data * other, parents=[self]) 
            
            def _backward():
                self.grad += other * out.grad
            
            out._backward = _backward
            
            return out
                    
        other = self._coerce(other)
    
        assert (
                self.data.shape == other.data.shape or # (m, n) * (m, n)
                    (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) # (m, n) * (n,)
                    ), "not supporting broadcasting for now" 
        out = Spyder(self.data * other.data, parents=[self, other])
    
        def _backward():
            if self.data.shape == other.data.shape:
                self.grad += other.data * out.grad
                other.grad += self.data * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += other.data * out.grad 
                other.grad += np.sum(self.data * out.grad, axis = 0) 
    
        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = self._coerce(other)

        return self * (other ** -1) # lol this is funny for some reason :sob:

    def __neg__(self):
        out = Spyder(-self.data, parents = [self])
        def _backward():
            self.grad -= out.grad
        out._backward = _backward
        return out

    def __rsub__(self, other):
        return -self + other 

    def __radd__(self, other):
        return self + other # addition is commutative 

    def __rmul__(self, other):
        return self * other 

    def __rtruediv__(self, other):
        return other * (self ** -1) 

    def __getitem__(self, idx):
        out = Spyder(self.data[idx], parents = [self])

        def _backward():
            self.grad[idx] += out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Spyder(np.maximum(0, self.data), parents = [self])

        def _backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = _backward
        return out

    def sigmoid(self):
        x = self.data

        out_data = np.empty_like(x)

        positive = x >= 0 # prepare boolean mask
        negative = ~positive

        out_data[positive] = 1 / (1 + np.exp(-x[positive])) 
        out_data[negative] = np.exp(x[negative]) / (1 + np.exp(x[negative]))

        out = Spyder(out_data, parents = [self])

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad

        out._backward = _backward
        return out

    def log(self):
        x = np.clip(self.data, 1e-8, None)
        out = Spyder(np.log(x), parents = [self]) # clip x value to make sure log doesnt go boom

        def _backward():
            self.grad += (self.data >= 1e-8) / x * out.grad 

        out._backward = _backward
        return out

    def exp(self):
        out = Spyder(np.exp(self.data), [self])

        def _backward():
            self.grad += np.exp(self.data) * out.grad 

        out._backward = _backward
        return out
    
    def retrace(self):
        self.clean_webs()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for v in reversed(topo): 
            v._backward()

    def clean_webs(self):
        visited = set()
        def visit_parents(v):
            if v not in visited:
                visited.add(v)
                v.grad = np.zeros_like(v.data)
                for parent in v.parents:
                    visit_parents(parent)
        visit_parents(self)

Binary Classification Multi-layer Neural Network

In [104]:
class Web: 
    def __init__(self, layer_size, learning_rate = 0.001, intialization_strength = 0.01, threshold = 0.5, random_state = 11, epochs = 1000):

        """
        x, y -> np.ndarray or spyder
        layer_size -> (input features, layer 1 neurons, layer 2 neurons, ..., outputs)
        """
        self.rng = np.random.default_rng(random_state)
        self.layer_size = layer_size
        self.learning_rate = learning_rate
        self.intialization_strength = intialization_strength
        self.epochs = epochs
        self.threshold = threshold

    def spin(self, x, y):

        x = x if isinstance(x, Spyder) else Spyder(x)
        y = y if isinstance(y, Spyder) else Spyder(y)

        self.weights = []
        self.biases = []

        for n in range(len(self.layer_size) - 1):
        
            in_features = self.layer_size[n]
            out_features = self.layer_size[n + 1]
        
            self.weights.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (in_features, out_features))))
            self.biases.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (out_features,))))

        for epoch in range(self.epochs):
            inp = x
            for w, b in zip(self.weights[:-1], self.biases[:-1]):
                inp = (inp @ w + b).sigmoid()

            pred = (inp @ self.weights[-1] + self.biases[-1]).sigmoid()
            error = (-(y * pred.log() + (1 - y) * (1 - pred).log())).mean() 
            
            error.retrace()

            for w, b in zip(self.weights, self.biases):
                w.data -= self.learning_rate * w.grad
                b.data -= self.learning_rate * b.grad

            if epoch % 50 == 0:
                print(f"epoch {epoch}, loss = {error.data}")

    def predict_proba(self, x):
        inp = x if isinstance(x, Spyder) else Spyder(x)
        for w, b in zip(self.weights[:-1], self.biases[:-1]):
            inp = (inp @ w + b).sigmoid()
        pred = inp @ self.weights[-1] + self.biases[-1]
        return pred

    def predict(self, x):
        return ((self.predict_proba(x)).sigmoid().data >= self.threshold)

In [105]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

In [106]:
x, y = load_breast_cancer(return_X_y=True, as_frame=True)

In [107]:
x_scaler = StandardScaler()

In [108]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size = 0.3,
    random_state = 11
)

In [109]:
x_train_scaled = x_scaler.fit_transform(x_train.to_numpy())
x_test_scaled = x_scaler.transform(x_test.to_numpy())

In [110]:
web = Web([x_train_scaled.shape[1], 20, 10, 5, 1], 1, intialization_strength=0.5, epochs = 2000)

In [111]:
web.spin(x_train_scaled, y_train.to_numpy().reshape(-1, 1))

epoch 0, loss = 0.7062516049102819
epoch 50, loss = 0.20518754759729085
epoch 100, loss = 0.07711035296000712
epoch 150, loss = 0.053038229110371524
epoch 200, loss = 0.03997321631886153
epoch 250, loss = 0.030566882391192176
epoch 300, loss = 0.024366350764342178
epoch 350, loss = 0.01981997399415064
epoch 400, loss = 0.01571925652872627
epoch 450, loss = 0.012079205300823764
epoch 500, loss = 0.009259008424297019
epoch 550, loss = 0.007250588672852939
epoch 600, loss = 0.005834975584254
epoch 650, loss = 0.004816516470533415
epoch 700, loss = 0.004063322373718228
epoch 750, loss = 0.0034910732697766827
epoch 800, loss = 0.0030455853558719772
epoch 850, loss = 0.00269128294100062
epoch 900, loss = 0.0024042021858870152
epoch 950, loss = 0.002167783900622917
epoch 1000, loss = 0.001970308157670181
epoch 1050, loss = 0.001803295978130452
epoch 1100, loss = 0.001660489762505682
epoch 1150, loss = 0.001537187501447055
epoch 1200, loss = 0.001429798206078227
epoch 1250, loss = 0.0013355388

In [112]:
pred = web.predict(x_test_scaled)

In [113]:
from sklearn.metrics import classification_report

In [114]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93        61
           1       0.96      0.96      0.96       110

    accuracy                           0.95       171
   macro avg       0.95      0.95      0.95       171
weighted avg       0.95      0.95      0.95       171



Verson 4: Axis Aware(mostly, not completely) Tensor Autograd (still supporting only certain broadcasting methods)

In [297]:
import numpy as np 
import numbers 
class Spyder:
    def __init__(self, data: np.ndarray, parents = None):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.parents = parents if parents else []
        self._backward = lambda: None

    def _coerce(self, other):
        """function to coerce other into a spyder"""
        if isinstance(other, Spyder):
            return other
        if isinstance(other, (numbers.Real, np.generic, np.ndarray)):
            return Spyder(other)
        raise TypeError(
                f"Received object of type {type(other)}, was unable to coerce to Spyder object"
                f"\nOnly supporting scalars, np.ndarrays and other spyders!"
                )
    def __repr__(self):
        return f"Spyder(data = {self.data}, grad = {self.grad}, parents = {self.parents})"

    def __matmul__(self, other):
        other = self._coerce(other)

        try:
            assert self.data.shape[1] == other.data.shape[0], "matrixes must be in shape (m, n), (n, p) for matmul"
        except IndexError:
            raise ValueError("Sorry bro i was too lazy to implement any other broadcasting type :(\n"
                            f"So i got a matrices of shapes {self.data.shape} and {other.data.shape}"
                            "it is not in required (m, n), (n, p) form so i cannot calculate sorry :((\n"
                            "just use tinygrad or micrograd bro i so sorry pls forgive me")
        out = Spyder(self.data @ other.data, parents = [self, other])

        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad

        out._backward = _backward
        return out

    def sum(self, axis = None, keepdims = False):
        out = Spyder(np.sum(self.data, axis = axis, keepdims=keepdims), parents = [self,])

        def _backward():
            if axis is None:
                self.grad += np.ones_like(self.data) * out.grad
                return
            grad = out.grad if keepdims else np.expand_dims(out.grad, axis=axis)    
            self.grad += np.broadcast_to(grad, self.data.shape)
                
        out._backward = _backward
        return out

    def mean(self, axis = None, keepdims = False):
        n = self.data.size if axis is None else np.size(self.data, axis = axis)
        return self.sum(axis = axis, keepdims = keepdims) / n

    def max(self, axis = None, keepdims = True):
        out_data = np.max(
            self.data,
            axis = axis,
            keepdims = keepdims
        )

        mask_data = out_data if keepdims else np.expand_dims(out_data, axis)

        out = Spyder(out_data, parents = [self])

        def _backward():
            mask = (self.data == mask_data)
            grad = out.grad if keepdims else np.expand_dims(out.grad, axis = axis)
            self.grad += mask * np.broadcast_to(grad, self.data.shape)

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert (isinstance(other, (numbers.Real, np.generic))), "only supporting int/float powers for now"

        out = Spyder(self.data ** other, parents = [self]) 

        def _backward():
            self.grad += (other * (self.data) ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):        
        other = self._coerce(other)

        assert (
            other.data.ndim == 0 or # (m, n) -+(m, n)
            self.data.shape == other.data.shape or # (m, n) + (m, n)
            (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) or # (m, n) + (n,)
            (self.data.shape[0] == other.data.shape[0] and other.data.ndim == 2 and other.data.shape[1] == 1) # (m, n) +(m, 1)
            ), f"Broadcasting arrays of shape {self.data.shape} and {other.data.shape} is not supported."
        
        out = Spyder(self.data + other.data, parents=[self, other])

        def _backward():

            if other.data.ndim == 0:
                self.grad += out.grad
                other.grad += np.sum(out.grad)

            elif self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad += np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad += np.sum(out.grad, axis = 0)

            elif (self.data.shape[0] == other.data.shape[0] and other.data.ndim == 2 and other.data.shape[1] == 1):
                self.grad += out.grad
                other.grad += np.sum(
                    out.grad,
                    axis = 1,
                    keepdims = True
                    )

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = self._coerce(other)

        assert (
            other.data.ndim == 0 or # (m, n) - q
            self.data.shape == other.data.shape or # (m, n) - (m, n)
            (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) or # (m, n) -(n,)
            (self.data.shape[0] == other.data.shape[0] and other.data.ndim == 2 and other.data.shape[1] == 1) # (m, n) - (m, 1)
            ), f"Broadcasting arrays of shape {self.data.shape} and {other.data.shape} is not supported."
        
        out = Spyder(self.data - other.data, parents=[self, other])

        def _backward():

            if other.data.ndim == 0:
                self.grad += out.grad 
                other.grad -= np.sum(out.grad)

            elif self.data.shape == other.data.shape: 
                self.grad += np.ones_like(self.data) * out.grad
                other.grad -= np.ones_like(other.data) * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += out.grad
                other.grad -= np.sum(out.grad, axis = 0)

            elif (self.data.shape[0] == other.data.shape[0] and other.data.ndim == 2 and other.data.shape[1] == 1):
                self.grad += out.grad
                other.grad -= np.sum(
                    out.grad,
                    axis = 1,
                    keepdims = True
                    )
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = self._coerce(other)
    
        assert (
            other.data.ndim == 0 or # (m, n) * q
            self.data.shape == other.data.shape or # (m, n) * (m, n)
            (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1) or # (m, n) * (n,) 
            (self.data.shape[0] == other.data.shape[0] and other.data.ndim == 2 and other.data.shape[1] == 1) # (m, n) * (m, 1)
            ), f"Broadcasting arrays of shape {self.data.shape} and {other.data.shape} is not supported."
  
        out = Spyder(self.data * other.data, parents=[self, other])
    
        def _backward():

            if other.data.ndim == 0:
                self.grad += other.data * out.grad
                other.grad += np.sum(self.data * out.grad)

            elif self.data.shape == other.data.shape:
                self.grad += other.data * out.grad
                other.grad += self.data * out.grad

            elif (self.data.shape[1] == other.data.shape[0] and other.data.ndim == 1):
                self.grad += other.data * out.grad 
                other.grad += np.sum(self.data * out.grad, axis = 0) 

            elif (self.data.shape[0] == other.data.shape[0] and other.data.ndim == 2 and other.data.shape[1] == 1):
                self.grad += np.broadcast_to(other.data, self.data.shape) * out.grad
                other.grad += np.sum(
                    self.data * out.grad,
                    axis = 1,
                    keepdims = True
                )
    
        out._backward = _backward
        return out

    def __truediv__(self, other):
        other = self._coerce(other)

        return self * (other ** -1) # lol this is funny for some reason :sob:

    def __neg__(self):
        out = Spyder(-self.data, parents = [self])
        def _backward():
            self.grad -= out.grad
        out._backward = _backward
        return out

    def __rsub__(self, other):
        return -self + other 

    def __radd__(self, other):
        return self + other # addition is commutative 

    def __rmul__(self, other):
        return self * other 

    def __rtruediv__(self, other):
        return other * (self ** -1) 

    def __getitem__(self, idx):
        out = Spyder(self.data[idx], parents = [self])

        def _backward():
            self.grad[idx] += out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Spyder(np.maximum(0, self.data), parents = [self])

        def _backward():
            self.grad += (self.data > 0) * out.grad

        out._backward = _backward
        return out

    def sigmoid(self):
        x = self.data

        out_data = np.empty_like(x)

        positive = x >= 0 # prepare boolean mask
        negative = ~positive

        out_data[positive] = 1 / (1 + np.exp(-x[positive])) 
        out_data[negative] = np.exp(x[negative]) / (1 + np.exp(x[negative]))

        out = Spyder(out_data, parents = [self])

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad

        out._backward = _backward
        return out

    def log(self):
        x = np.clip(self.data, 1e-8, None)
        out = Spyder(np.log(x), parents = [self]) # clip x value to make sure log doesnt go boom

        def _backward():
            mask = (self.data >= 1e-8)
            self.grad += mask * (1 / x) * out.grad 

        out._backward = _backward
        return out

    def exp(self):
        out_data = np.exp(self.data)
        out = Spyder(out_data, [self])

        def _backward():
            self.grad += out_data * out.grad 

        out._backward = _backward
        return out
    
    def retrace(self):
        self.clean_webs()
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parents:
                    build_topo(parent)
                topo.append(v)
        build_topo(self)

        self.grad = np.ones_like(self.data)
        for v in reversed(topo): 
            v._backward()

    def clean_webs(self):
        visited = set()
        def visit_parents(v):
            if v not in visited:
                visited.add(v)
                v.grad = np.zeros_like(v.data)
                for parent in v.parents:
                    visit_parents(parent)
        visit_parents(self)

In [298]:
x = np.array([[[1, 2, 5],
              [3, 4, 6],
              [1, 2, 5],
              [3, 4, 6]]])
axis = 2

In [299]:
x.sum(axis), np.size(x, axis=axis)

(array([[ 8, 13,  8, 13]]), 3)

Multi-class Classification Multi-layer Neural Network

In [312]:
import numpy as np
class Web: 
    def __init__(self, layer_size, learning_rate = 0.001, intialization_strength = 0.01, random_state = 11, epochs = 1000):

        """
        x, y -> np.ndarray 
        x.shape = (samples, features)
        y.shape = (samples,)
        target is expected to be label encoded.
        layer_size -> (input features, layer 1 neurons, layer 2 neurons, ..., outputs)
        """
        self.rng = np.random.default_rng(random_state)
        self.layer_size = layer_size
        self.learning_rate = learning_rate
        self.intialization_strength = intialization_strength
        self.epochs = epochs

    def spin(self, x, y):

        y = np.expand_dims(y, axis = 1)
        self.distribution = np.unique(y)
        y = np.array(self.distribution == y, dtype = float)

        x = x if isinstance(x, Spyder) else Spyder(x)
        y = y if isinstance(y, Spyder) else Spyder(y)

        self.weights = []
        self.biases = []

        for n in range(len(self.layer_size) - 1):
        
            in_features = self.layer_size[n]
            out_features = self.layer_size[n + 1]
        
            self.weights.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (in_features, out_features))))
            self.biases.append(Spyder(self.rng.normal(loc = 0, scale = self.intialization_strength, size = (out_features,))))

        for epoch in range(self.epochs):
            inp = x
            for w, b in zip(self.weights[:-1], self.biases[:-1]):
                inp = (inp @ w + b).relu()

            logits = (inp @ self.weights[-1] + self.biases[-1])
            logits = logits - logits.max(axis = 1, keepdims = True)
            pred = logits.exp() / logits.exp().sum(axis = 1, keepdims = True)
            error = (-(y * pred.log())).sum(axis = 1, keepdims = True).mean()
            
            error.retrace()

            for w, b in zip(self.weights, self.biases):
                w.data -= self.learning_rate * w.grad
                b.data -= self.learning_rate * b.grad

            if epoch % 50 == 0:
                print(f"epoch {epoch}, loss = {error.data}")

    def predict_proba(self, x):
        inp = x if isinstance(x, Spyder) else Spyder(x) # coerce inputs to spyder

        for w, b in zip(self.weights[:-1], self.biases[:-1]):
            inp = (inp @ w + b).relu() # relu activation for hidden layers

        logits = (inp @ self.weights[-1] + self.biases[-1]) # logits from final layer
        logits = logits - logits.max(axis = 1, keepdims = True)  # shift logits before .exp() to get smaller values

        pred = logits.exp() / logits.exp().sum(axis = 1, keepdims = True) # softmax
        
        return pred  # return probability distribution

    def predict(self, x):
        return self.distribution[np.argmax(self.predict_proba(x).data, axis = 1)]

In [313]:
y = np.expand_dims(np.array([1, 4, 5, 2]), axis = 1)
distribution = np.unique(y)

In [314]:
np.array(distribution == y, dtype = float)

array([[1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.]])

In [315]:
from sklearn.datasets import load_iris

In [316]:
x, y = load_iris(return_X_y=True, as_frame=True)

In [317]:
x.head(), y.head()

(   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
 0                5.1               3.5                1.4               0.2
 1                4.9               3.0                1.4               0.2
 2                4.7               3.2                1.3               0.2
 3                4.6               3.1                1.5               0.2
 4                5.0               3.6                1.4               0.2,
 0    0
 1    0
 2    0
 3    0
 4    0
 Name: target, dtype: int64)

In [318]:
x.size

600

In [319]:
from sklearn.preprocessing import StandardScaler

In [320]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size = 0.33,
    random_state = 11
)

In [321]:
x_scaler = StandardScaler()
x_train, x_test = x_scaler.fit_transform(x_train.to_numpy()), x_scaler.transform(x_test.to_numpy())

In [326]:
web = Web([x_train.shape[1], 10, 5, np.unique(y_train).size], learning_rate = 0.5, intialization_strength = 0.01, epochs = 1500)

In [327]:
web.spin(x_train, y_train.to_numpy())

epoch 0, loss = 1.0986089971956827
epoch 50, loss = 1.0982074299398528
epoch 100, loss = 1.0982036186052506
epoch 150, loss = 1.0981882063035908
epoch 200, loss = 1.0980585470185973
epoch 250, loss = 1.0725815163953194
epoch 300, loss = 0.22101987449173727
epoch 350, loss = 0.08090392924221432
epoch 400, loss = 0.04949963419898033
epoch 450, loss = 0.03704422779313332
epoch 500, loss = 0.029937717274158615
epoch 550, loss = 0.02502511706089983
epoch 600, loss = 0.021242002802652156
epoch 650, loss = 0.01815250078119862
epoch 700, loss = 0.01549722515363345
epoch 750, loss = 0.013256900567108529
epoch 800, loss = 0.011376720690240785
epoch 850, loss = 0.009816951032978886
epoch 900, loss = 0.008520573182139954
epoch 950, loss = 0.007441194061980562
epoch 1000, loss = 0.006541555839099162
epoch 1050, loss = 0.005787481689203816
epoch 1100, loss = 0.0051516109634932574
epoch 1150, loss = 0.004612679641486004
epoch 1200, loss = 0.004153007813425808
epoch 1250, loss = 0.0037588654311249197


In [328]:
pred = web.predict(x_train)

print("accuracy:", np.mean(pred == y_train.to_numpy()))

accuracy: 1.0


In [329]:
pred = web.predict(x_test)

print("test accuracy:", np.mean(pred == y_test.to_numpy()))

test accuracy: 0.96
